In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


مرحلة بناء مدير الحالة واختباره

In [ ]:
%%writefile /content/drive/MyDrive/Adaptive_Tutor_RL/src/state_manager/session_tracker.py
import pandas as pd
import random

class SessionTracker:
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path)
        self.current_level = 'A1'
        self.consecutive_knows = 0
        self.history = []

    def get_word(self):
        # تصفية الكلمات بناءً على المستوى الحالي
        level_words = self.df[self.df['level'] == self.current_level]
        if level_words.empty:
            return None
        return random.choice(level_words['word'].values)

    def process_response(self, word, knows):
        if knows:
            self.consecutive_knows += 1
            if self.consecutive_knows >= 3:
                self.upgrade_level()
        else:
            self.consecutive_knows = 0
            return "TRIGGER_NEURAL_MODEL"
        return "CONTINUE"

    def upgrade_level(self):
        levels = ['A1', 'A2', 'B1', 'B2', 'C1', 'C2']
        current_idx = levels.index(self.current_level)
        if current_idx < len(levels) - 1:
            self.current_level = levels[current_idx + 1]
            self.consecutive_knows = 0
            print(f"🎉 المستوى تمت ترقيته إلى: {self.current_level}")

Writing /content/drive/MyDrive/Adaptive_Tutor_RL/src/state_manager/session_tracker.py


In [ ]:
import sys
sys.path.append('/content/drive/MyDrive/Adaptive_Tutor_RL/src')
from state_manager.session_tracker import SessionTracker

# تهيئة مدير الحالة
tracker = SessionTracker('/content/drive/MyDrive/Adaptive_Tutor_RL/data/processed/repo_words.csv')

print(f"بدء الجلسة في المستوى: {tracker.current_level}")

# محاكاة: الطالب يعرف 3 كلمات (ليتم ترقيته)
for i in range(3):
    word = tracker.get_word()
    print(f"السؤال: هل تعرف معنى كلمة '{word}'؟ (نعم/لا)")
    # سنفترض هنا أن الطالب أجاب بـ "نعم"
    result = tracker.process_response(word, knows=True)
    print(f"الحالة: {result}")

# محاكاة: الطالب لا يعرف كلمة
word = tracker.get_word()
print(f"السؤال: هل تعرف معنى كلمة '{word}'؟")
result = tracker.process_response(word, knows=False)
print(f"الحالة: {result}") # يجب أن يطبع هنا TRIGGER_NEURAL_MODEL

بدء الجلسة في المستوى: A1
السؤال: هل تعرف معنى كلمة 'breakfast'؟ (نعم/لا)
الحالة: CONTINUE
السؤال: هل تعرف معنى كلمة 'welcome'؟ (نعم/لا)
الحالة: CONTINUE
السؤال: هل تعرف معنى كلمة 'downstairs'؟ (نعم/لا)
🎉 المستوى تمت ترقيته إلى: A2
الحالة: CONTINUE
السؤال: هل تعرف معنى كلمة 'biology'؟
الحالة: TRIGGER_NEURAL_MODEL


المرحلة الثانية " التدريب "


انشاء المجلدات والتحويل والتقسيم

تجهيز بيانات SFT

In [ ]:
import os

# تعريف المسار الرئيسي للمشروع
base_path = '/content/drive/MyDrive/Adaptive_Tutor_RL'

# قائمة المجلدات الفرعية المطلوبة
subdirectories = [
    'src/state_manager',
    'src/sft',
    'src/rewards',
    'src/rl',
    'data/processed'
]

# إنشاء المجلدات
for sub in subdirectories:
    full_path = os.path.join(base_path, sub)
    os.makedirs(full_path, exist_ok=True)
    print(f"✅ تم التأكد من وجود المجلد: {full_path}")

✅ تم التأكد من وجود المجلد: /content/drive/MyDrive/Adaptive_Tutor_RL/src/state_manager
✅ تم التأكد من وجود المجلد: /content/drive/MyDrive/Adaptive_Tutor_RL/src/sft
✅ تم التأكد من وجود المجلد: /content/drive/MyDrive/Adaptive_Tutor_RL/src/rewards
✅ تم التأكد من وجود المجلد: /content/drive/MyDrive/Adaptive_Tutor_RL/src/rl
✅ تم التأكد من وجود المجلد: /content/drive/MyDrive/Adaptive_Tutor_RL/data/processed


In [ ]:
%%writefile /content/drive/MyDrive/Adaptive_Tutor_RL/src/sft/prepare_dataset.py
import json
import random
import os
from transformers import AutoTokenizer

def clean_and_format_examples(examples_list):
    """تنظيف قائمة الأمثلة وتحويلها إلى نص منقط ومفهوم للنماذج الصغيرة"""
    if not examples_list:
        return "No examples provided."
    cleaned_lines = []
    for i, ex in enumerate(examples_list, 1):
        clean_ex = str(ex).strip().replace('\u2009', ' ')
        cleaned_lines.append(f"  {i}. {clean_ex}")
    return "\n".join(cleaned_lines)

def transform_shuffle_and_split_qwen(input_path, train_out_path, val_out_path, test_out_path, train_ratio=0.80, val_ratio=0.10):
    if not os.path.exists(input_path):
        print(f"❌ خطأ: الملف غير موجود في المسار: {input_path}")
        return

    print("📥 Loading tokenizer to format prompt/completion pairs natively...")
    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen1.5-1.8B-Chat", trust_remote_code=True)

    formatted_dataset = []

    with open(input_path, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            try:
                data = json.loads(line.strip())
                word = data.get("word", "").strip()
                w_type = data.get("type", "").strip()
                level = data.get("level", "").strip()
                examples = data.get("examples", [])
                dialogue = data.get("dialogue", "").strip()

                formatted_examples = clean_and_format_examples(examples)

                user_content = (
                    f"### WORD DATA\n"
                    f"- Target Word: {word}\n"
                    f"- Part of Speech: {w_type}\n"
                    f"- CEFR Proficiency Level: {level}\n"
                    f"- Contextual Reference Examples:\n{formatted_examples}\n\n"
                    f"### PEDAGOGICAL TASK\n"
                    f"Act as an adaptive tutor. Initiate the structured dialogue from the training data to teach the student the word '{word}' naturally, adhering to the provided CEFR level constraints."
                )

                messages = [
                    {"role": "system", "content": "You are an expert, patient, and adaptive English language tutor."},
                    {"role": "user", "content": user_content},
                    {"role": "assistant", "content": dialogue}
                ]

                # تطبيق قالب Qwen للحصول على النص الكامل للمحادثة
                full_chat = tokenizer.apply_chat_template(messages, tokenize=False)

                # فصل النص بدقة رياضية عند بداية استجابة المساعد (Assistant)
                target_boundary = "<|im_start|>assistant\n"
                if target_boundary in full_chat:
                    parts = full_chat.split(target_boundary, 1)
                    prompt = parts[0] + target_boundary
                    completion = parts[1]
                else:
                    # ميكانيكية احتياطية في حال تغير القالب
                    prompt = tokenizer.apply_chat_template(messages[:-1], tokenize=False) + target_boundary
                    completion = dialogue + tokenizer.eos_token

                # بناء الهيكل الحديث المعتمد للـ SFT الموجه
                formatted_entry = {
                    "prompt": prompt,
                    "completion": completion
                }
                formatted_dataset.append(formatted_entry)
            except Exception as e:
                print(f"⚠️ خطأ في معالجة السطر: {e}")
                continue

    print(f"📦 إجمالي العينات المعالجة والمحسنة لـ Qwen: {len(formatted_dataset)}")

    random.seed(42)
    random.shuffle(formatted_dataset)

    total = len(formatted_dataset)
    train_end = int(total * train_ratio)
    val_end = train_end + int(total * val_ratio)

    train_data = formatted_dataset[:train_end]
    val_data = formatted_dataset[train_end:val_end]
    test_data = formatted_dataset[val_end:]

    for path, data in [(train_out_path, train_data), (val_out_path, val_data), (test_out_path, test_data)]:
        with open(path, 'w', encoding='utf-8') as f:
            for entry in data:
                f.write(json.dumps(entry, ensure_ascii=False) + '\n')

    print(f"✅ تم الحفظ بنجاح وتوليد المجموعات الثلاث بنسق (Prompt/Completion) المعتمد حديثاً.")

if __name__ == "__main__":
    transform_shuffle_and_split_qwen(
        input_path='/content/drive/MyDrive/Adaptive_Tutor_RL/data/processed/final_merged_dataset.jsonl',
        train_out_path='/content/drive/MyDrive/Adaptive_Tutor_RL/data/processed/train_sft.jsonl',
        val_out_path='/content/drive/MyDrive/Adaptive_Tutor_RL/data/processed/val_sft.jsonl',
        test_out_path='/content/drive/MyDrive/Adaptive_Tutor_RL/data/processed/test_sft.jsonl'
    )

Overwriting /content/drive/MyDrive/Adaptive_Tutor_RL/src/sft/prepare_dataset.py


In [ ]:
!python /content/drive/MyDrive/Adaptive_Tutor_RL/src/sft/prepare_dataset.py

📥 Loading tokenizer to format prompt/completion pairs natively...
config.json: 100% 662/662 [00:00<00:00, 3.25MB/s]
tokenizer_config.json: 100% 1.29k/1.29k [00:00<00:00, 4.03MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 76.5MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 98.0MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 131MB/s]
📦 إجمالي العينات المعالجة والمحسنة لـ Qwen: 1000
✅ تم الحفظ بنجاح وتوليد المجموعات الثلاث بنسق (Prompt/Completion) المعتمد حديثاً.


In [ ]:
# 1. تشغيل السكربت المحدث
!python /content/drive/MyDrive/Adaptive_Tutor_RL/src/sft/prepare_dataset.py

# 2. فحص النتيجة الحقيقية بعد التنظيف والترتيب
import json

print("\n--- 🔍 معاينة الشكل الهيكلي الجديد المحسن لنماذج Qwen ---")
with open('/content/drive/MyDrive/Adaptive_Tutor_RL/data/processed/test_sft.jsonl', 'r', encoding='utf-8') as f:
    sample_entry = json.loads(f.readline())
    print(json.dumps(sample_entry, indent=2, ensure_ascii=False))

📦 إجمالي العينات المعالجة والمحسنة لـ Qwen: 1000
✅ تم الحفظ بنجاح وتوليد المجموعات الثلاث بنسق مهيكل.

--- 🔍 معاينة الشكل الهيكلي الجديد المحسن لنماذج Qwen ---
{
  "messages": [
    {
      "role": "system",
      "content": "You are an expert, patient, and adaptive English language tutor."
    },
    {
      "role": "user",
      "content": "### WORD DATA\n- Target Word: steal\n- Part of Speech: verb\n- CEFR Proficiency Level: A2\n- Contextual Reference Examples:\n  1. I'll report you to the police if I catch you stealing again.\n  2. steal from somebody/something We found out he'd been stealing from us for years.\n  3. steal something My wallet was stolen.\n  4. I had my wallet stolen.\n  5. Thieves stole jewellery worth over £10 000.\n  6. steal something from somebody/something He stole a car from the parking lot of a mall.\n  7. It's a crime to handle stolen goods.\n  8. He was charged with possession of stolen property.\n  9. (figurative) to steal somebody’s ideas\n  10. (figurat

برمجة ملف التدريب الأساسي train_qwen.py


In [ ]:
!pip install -q torch transformers datasets trl peft bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.9 MB/s eta 0:00:00


In [ ]:
!pip install -U trl transformers peft bitsandbytes accelerate

In [ ]:
import trl
print(dir(trl))

['BEMACallback', 'DPOConfig', 'DPOTrainer', 'DatasetMixtureConfig', 'GRPOConfig', 'GRPOTrainer', 'KTOConfig', 'KTOTrainer', 'LogCompletionsCallback', 'ModelConfig', 'RLOOConfig', 'RLOOTrainer', 'RewardConfig', 'RewardTrainer', 'RichProgressCallback', 'SFTConfig', 'SFTTrainer', 'ScriptArguments', 'SyncRefModelCallback', 'TrlParser', 'WeaveCallback', '__all__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '_class_to_module', '_import_structure', '_modules', '_name', '_objects', 'add_response_schema', 'apply_chat_template', 'chat_template_utils', 'clone_chat_template', 'create_reference_model', 'data_utils', 'extract_prompt', 'get_dataset', 'get_kbit_device_map', 'get_peft_config', 'get_quantization_config', 'get_training_chat_template', 'init_zero_verbose', 'is_conversational', 'is_conversational_from_value', 'maybe_apply_chat_template', 'maybe_convert_to_chatml', 'maybe_extract_prompt', 'maybe_unpair_preference_dataset', 'models', 'pack_dataset

In [ ]:
%%writefile /content/drive/MyDrive/Adaptive_Tutor_RL/src/sft/train_qwen.py
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)
from peft import LoraConfig, TaskType

# استيراد المكونات الأساسية والمستقرة والمضمونة 100% عبر كافة الإصدارات
from trl import SFTConfig, SFTTrainer

def run_sft():
    model_id = "Qwen/Qwen1.5-1.8B-Chat"
    output_dir = "/content/drive/MyDrive/Adaptive_Tutor_RL/models/qwen_sft_checkpoint"

    # 1. إعدادات الضغط 4-bit للذاكرة متوافقة أصلياً مع bfloat16
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, # التحديث لـ bfloat16 لتوحيد الحسابات
        bnb_4bit_use_double_quant=True
    )

    # 2. تحميل المحلل والنموذج بصيغة bfloat16 الأصلية له
    print(f"📥 Loading model and tokenizer in native bfloat16: {model_id}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.bfloat16  # التحميل بالصيغة الأصيلة المستقرة للنموذج
    )

    # 3. تحميل البيانات (التي تحتوي على prompt و completion)
    print("📦 Loading datasets from Drive...")
    data_files = {
        "train": "/content/drive/MyDrive/Adaptive_Tutor_RL/data/processed/train_sft.jsonl",
        "validation": "/content/drive/MyDrive/Adaptive_Tutor_RL/data/processed/val_sft.jsonl"
    }
    dataset = load_dataset("json", data_files=data_files)

    # 4. إعدادات LoRA للأوزان
    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM
    )

    # 5. المعايير الحديثة والمستقرة للتدريب باستخدام SFTConfig
    training_args = SFTConfig(
        output_dir=output_dir,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        gradient_checkpointing=True,
        optim="paged_adamw_32bit",
        max_grad_norm=0.3,
        learning_rate=2e-4,
        warmup_steps=10,
        num_train_epochs=3,
        eval_strategy="steps",
        eval_steps=50,
        logging_steps=10,
        save_strategy="steps",
        save_steps=100,
        save_total_limit=2,
        fp16=False,                         # تعطيل fp16 تماماً لإيقاف الـ GradScaler المتسبب بالخطأ
        bf16=True,                          # تفعيل التدريب الأصيل المستقر بنمط bf16
        report_to="none",
        completion_only_loss=True,
        dataset_text_field=None,
        max_length=1024
    )

    # 6. تهيئة المدرب تلقائياً
    trainer = SFTTrainer(
        model=model,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        peft_config=peft_config,
        processing_class=tokenizer,
        args=training_args
    )

    # ميكانيكية الاستئناف الآمن من الكراش
    resume_from_checkpoint = None
    if os.path.exists(output_dir) and any(f.startswith("checkpoint") for f in os.listdir(output_dir)):
        print("🔄 Found existing checkpoint, resuming training...")
        resume_from_checkpoint = True

    print("🔥 Starting SFT training execution (Native BF16 Mode)...")
    trainer.train(resume_from_checkpoint=resume_from_checkpoint)

    # 7. حفظ المخرجات النهائية
    final_dir = "/content/drive/MyDrive/Adaptive_Tutor_RL/models/qwen_sft_final"
    trainer.model.save_pretrained(final_dir)
    tokenizer.save_pretrained(final_dir)
    print("🎉 SFT Training Completed Successfully!")

if __name__ == "__main__":
    run_sft()

Overwriting /content/drive/MyDrive/Adaptive_Tutor_RL/src/sft/train_qwen.py


In [ ]:
!python /content/drive/MyDrive/Adaptive_Tutor_RL/src/sft/train_qwen.py

📥 Loading model and tokenizer in native bfloat16: Qwen/Qwen1.5-1.8B-Chat...
config.json: 100% 662/662 [00:00<00:00, 1.65MB/s]
tokenizer_config.json: 100% 1.29k/1.29k [00:00<00:00, 1.81MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 28.0MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 43.7MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 43.8MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100% 3.67G/3.67G [00:33<00:00, 109MB/s] 
Loading weights:   1% 2/291 [00:04<10:27,  2.17s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 291/291 [00:10<00:00, 27.40it/s]
generation_config.json: 100% 206/206 [00:00<00:00, 885kB/s]
📦 Loading datasets from Drive...
Generating train split: 800 examples [00:00, 68524.58 examples/s]
Gene

نهاية عملية تدريب SFT

اختبار عملية التدريب الجديدة باعددات محسنة مثل زيادة عدد الحقب , والحفظ التلقائي عند افضل اداء

In [ ]:
%%writefile /content/drive/MyDrive/Adaptive_Tutor_RL/src/sft/train_qwen_b.py
import os
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    EarlyStoppingCallback
)
from peft import LoraConfig, TaskType
from trl import SFTConfig, SFTTrainer

def run_sft_b():
    model_id = "Qwen/Qwen1.5-1.8B-Chat"

    # [تعديل] عزل مسار الـ Checkpoints للتجربة الحالية (b) عن التجربة الأولى
    checkpoint_dir = "/content/drive/MyDrive/Adaptive_Tutor_RL/models/qwen_sft_checkpoint_b"
    # مسار حفظ أفضل أوزان مستقرة مستهدفة
    final_best_dir = "/content/drive/MyDrive/Adaptive_Tutor_RL/models/qwen_sft_best_weight"

    # 1. إعدادات الضغط 4-bit للذاكرة متوافقة أصلياً مع bfloat16
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True
    )

    # 2. تحميل المحلل والنموذج بصيغة bfloat16 الأصلية
    print(f"📥 Loading model and tokenizer in native bfloat16: {model_id}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.bfloat16
    )

    # 3. تحميل البيانات من الـ Drive
    print("📦 Loading datasets from Drive...")
    data_files = {
        "train": "/content/drive/MyDrive/Adaptive_Tutor_RL/data/processed/train_sft.jsonl",
        "validation": "/content/drive/MyDrive/Adaptive_Tutor_RL/data/processed/val_sft.jsonl"
    }
    dataset = load_dataset("json", data_files=data_files)

    # 4. إعدادات LoRA للأوزان
    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM
    )

    # 5. صياغة الإعدادات الاحترافية الشاملة (Best Practices)
    training_args = SFTConfig(
        output_dir=checkpoint_dir,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        gradient_checkpointing=True,
        optim="paged_adamw_32bit",
        max_grad_norm=0.3,
        learning_rate=2e-4,
        warmup_steps=10,
        num_train_epochs=5,                  # 5 حقب تدريبية لمنح النموذج فرصة استكشاف أعلى لـ Loss أقل

        # استراتيجية الفحص والحفظ المتطورة
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=50,

        load_best_model_at_end=True,        # العودة التلقائية لأفضل أوزان تم رصدها بناءً على خسارة التحقق
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        # [تعديل] رفع حد الحفظ الاحتياطي لـ 2 للاحتفاظ بأفضل محطة والمحطة الحالية لضمان الأمان الفني
        save_total_limit=2,

        bf16=True,
        report_to="none",
        completion_only_loss=True,
        dataset_text_field=None,
        max_length=1024
    )

    # 6. تهيئة المدرب مع حقن خاصية الـ Early Stopping
    trainer = SFTTrainer(
        model=model,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        peft_config=peft_config,
        processing_class=tokenizer,
        args=training_args,
        callbacks=[
            # التوقف إذا مرت 3 فحوصات متتالية (150 خطوة) دون رصد أي تحسن في الـ eval_loss لمنع الـ Overfitting
            EarlyStoppingCallback(early_stopping_patience=3)
        ]
    )

    # ميكانيكية الاستئناف الآمن من الكراش للمجلد الجديد
    resume_from_checkpoint = None
    if os.path.exists(checkpoint_dir) and any(f.startswith("checkpoint") for f in os.listdir(checkpoint_dir)):
        print("🔄 Found existing checkpoint in directory (b), resuming training...")
        resume_from_checkpoint = True

    print("🔥 Starting Professional SFT Training Run [B]...")
    trainer.train(resume_from_checkpoint=resume_from_checkpoint)

    # 7. [تعديل واحترافي] حفظ الأوزان المثالية المنتقاة والمحلل دفعة واحدة باستخدام الطريقة المدمجة للـ Trainer
    print(f"💾 Saving the ABSOLUTE BEST model and tokenizer via trainer.save_model to: {final_best_dir}...")
    trainer.save_model(final_best_dir)
    print("🎉 SFT Training Run [B] Completed and Archived Successfully!")

if __name__ == "__main__":
    run_sft_b()

In [ ]:
!python /content/drive/MyDrive/Adaptive_Tutor_RL/src/sft/train_qwen_b.py

اختبار النموذج المدرب الاولى للتأكد منه

البدء بعملية الاختبار

In [ ]:
%%writefile /content/drive/MyDrive/Adaptive_Tutor_RL/src/sft/test_qwen_sft.py
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

def launch_inference():
    base_model_id = "Qwen/Qwen1.5-1.8B-Chat"
    # مسار مخرجات عملية التدريب الأولى التي نريد تقييمها
    adapter_dir = "/content/drive/MyDrive/Adaptive_Tutor_RL/models/qwen_sft_final"

    print("📥 Loading Tokenizer and Base Model...")
    tokenizer = AutoTokenizer.from_pretrained(adapter_dir, trust_remote_code=True)

    # تحميل النموذج الأساسي بنفس إعدادات الضغط وبصيغة bfloat16 المستقرة
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True
    )

    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        dtype=torch.bfloat16
    )

    print("🔗 Fusing LoRA Weights with Base Model...")
    # دمج الأوزان التي تدربت في المرحلة الأولى فوق النموذج الأساسي
    model = PeftModel.from_pretrained(base_model, adapter_dir)
    model.eval() # وضعية التقييم (تغلق الـ Dropout وتثبت الأوزان)

    print("\n🤖 Tutor Model is Ready! Type 'exit' to stop.\n" + "="*50)

    # حلقة تفاعلية لطرح الأسئلة على المعلم الذكي
    while True:
        user_input = input("🧠 Enter a prompt/question for the Tutor: ")
        if user_input.lower() == 'exit':
            print("Goodbye!")
            break

        if not user_input.strip():
            continue

        # صياغة النص المستهلك؛ (ملاحظة: إذا كنت تستخدم نظام Chat Templates يفضل تطبيقه هنا)
        # سنمرر النص بشكل مباشر ومطابق لطبيعة بيانات التدريب الخاصة بك
        inputs = tokenizer(user_input, return_tensors="pt").to("cuda")

        print("\n🔮 Tutor Generation:")
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,        # طول الإجابة الأقصى
                temperature=0.7,           # درجة الإبداع والتنوع (تتناغم مع الـ Entropy المستقرة لدينا)
                top_p=0.9,                 # فلترة الكلمات الأكثر منطقية
                do_sample=True,            # تفعيل أخذ العينات العشوائية الذكية
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id else tokenizer.eos_token_id
            )

        # فك تشفير المخرجات وعرض الإجابة الجديدة فقط
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        # إزالة نص السؤال الأصلي من العرض إذا ظهر في المخرجات
        if generated_text.startswith(user_input):
            generated_text = generated_text[len(user_input):].strip()

        print(generated_text)
        print("\n" + "="*50 + "\n")

if __name__ == "__main__":
    launch_inference()

Writing /content/drive/MyDrive/Adaptive_Tutor_RL/src/sft/test_qwen_sft.py


In [ ]:
!python /content/drive/MyDrive/Adaptive_Tutor_RL/src/sft/test_qwen_sft.py

📥 Loading Tokenizer and Base Model...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights:   1% 2/291 [00:04<11:36,  2.41s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 291/291 [00:14<00:00, 20.07it/s]
🔗 Fusing LoRA Weights with Base Model...

🤖 Tutor Model is Ready! Type 'exit' to stop.
🧠 Enter a prompt/question for the Tutor: hi

🔮 Tutor Generation:
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Hello! How can I help you today? Is there something specific you'd like to know or discuss? I'm here to answer any questions you might

In [ ]:
%%writefile /content/drive/MyDrive/Adaptive_Tutor_RL/src/sft/test_qwen_sft.py
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

def launch_inference():
    base_model_id = "Qwen/Qwen1.5-1.8B-Chat"
    adapter_dir = "/content/drive/MyDrive/Adaptive_Tutor_RL/models/qwen_sft_final"

    print("📥 Loading Tokenizer and Base Model...")
    tokenizer = AutoTokenizer.from_pretrained(adapter_dir, trust_remote_code=True)

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True
    )

    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.bfloat16
    )

    print("🔗 Fusing LoRA Weights with Base Model...")
    model = PeftModel.from_pretrained(base_model, adapter_dir)
    model.eval()

    print("\n🤖 Tutor Model is Ready (ChatML Template Enabled)! Type 'exit' to stop.\n" + "="*50)

    while True:
        user_input = input("🧠 Enter a prompt/question for the Tutor: ")
        if user_input.lower() == 'exit':
            print("Goodbye!")
            break

        if not user_input.strip():
            continue

        # -------------------------------------------------------------
        # 🔥 التعديل الجوهري: صياغة الرسالة داخل نظام القوالب الرسمي لـ Qwen
        # -------------------------------------------------------------
        messages = [
            {"role": "system", "content": "You are a helpful, adaptive AI tutor."},
            {"role": "user", "content": user_input}
        ]

        # تحويل المصفوفة إلى نص مهيكل بالرموز المخصصة <|im_start|> تلقائياً
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer([text], return_tensors="pt").to("cuda")

        print("\n🔮 Tutor Generation:")
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,        # زيادة المساحة للإجابات التعليمية الشاملة
                temperature=0.5,           # خفض الحرارة قليلاً ليكون أكثر رصانة وأقل هلوسة
                top_p=0.9,
                repetition_penalty=1.1,    # عقوبة التكرار لمنعه من إعادة الجمل أو الرموز
                do_sample=True,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id else tokenizer.eos_token_id
            )

        # فك التشفير وعرض الإجابة الصافية
        generated_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

        print(generated_text.strip())
        print("\n" + "="*50 + "\n")

if __name__ == "__main__":
    launch_inference()

Overwriting /content/drive/MyDrive/Adaptive_Tutor_RL/src/sft/test_qwen_sft.py


In [ ]:
!python /content/drive/MyDrive/Adaptive_Tutor_RL/src/sft/test_qwen_sft.py

📥 Loading Tokenizer and Base Model...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights:   1% 2/291 [00:00<01:20,  3.59it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 291/291 [00:11<00:00, 25.49it/s]
🔗 Fusing LoRA Weights with Base Model...

🤖 Tutor Model is Ready (ChatML Template Enabled)! Type 'exit' to stop.
🧠 Enter a prompt/question for the Tutor: hello, can you tell me about "thesis" meaning

🔮 Tutor Generation:
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
A thesis is a statement or argument that is presented in a formal

يجب ان نعيد التدريب واجبار النموذج على اللغة الانكليزية فقط وان يتذكر سجل المحادثة ثم تدريبه من جديد

فحص البيانات الاساسية قبل التدريب

In [ ]:
import json
import os

def inspect_and_format_sample(jsonl_path):
    # صمام أمان للتأكد من وجود الملف في الـ Drive أولاً
    if not os.path.exists(jsonl_path):
        print(f"❌ خطأ: لم يتم العثور على الملف في المسار المحدد:\n {jsonl_path}\nيرجى التأكد من ربط الـ Drive بشكل صحيح.")
        return

    print("🧐 جاري فحص وتحليل عينة من ملف البيانات الخاص بك...\n" + "="*50)

    with open(jsonl_path, 'r', encoding='utf-8') as f:
        first_line = f.readline()
        if not first_line.strip():
            print("⚠️ تنبيه: السطر الأول في الملف فارغ!")
            return
        data_item = json.loads(first_line)

    print(f"🔑 الكلمة المستهدفة: {data_item.get('word')} ({data_item.get('type')})")
    print(f"📊 المستوى التعليمي المستهدف (CEFR): {data_item.get('level')}\n")

    # تفكيك النص المسطح وتحويله ديناميكياً إلى هيكل الدردشة المتوافق مع النماذج
    raw_dialogue = data_item.get('dialogue', '')
    turns = raw_dialogue.split("\n\n")

    formatted_messages = [
        {"role": "system", "content": f"You are an expert AI Tutor helping a student learn the word '{data_item.get('word')}' at a {data_item.get('level')} level."}
    ]

    for turn in turns:
        if turn.startswith("Teacher:"):
            content = turn.replace("Teacher:", "").strip()
            formatted_messages.append({"role": "assistant", "content": content})
        elif turn.startswith("Student:"):
            content = turn.replace("Student:", "").strip()
            formatted_messages.append({"role": "user", "content": content})

    print("🛠️ الشكل الصحيح الذي يجب أن يرى به المدرب (SFTTrainer) هذه العينة:")
    print(json.dumps(formatted_messages, indent=2, ensure_ascii=False))

# 🔥 تفعيل وتشغيل الدالة فوراً على ملفك
target_path = "/content/drive/MyDrive/Adaptive_Tutor_RL/data/processed/train_sft.jsonl"
inspect_and_format_sample(target_path)

🧐 جاري فحص وتحليل عينة من ملف البيانات الخاص بك...
🔑 الكلمة المستهدفة: None (None)
📊 المستوى التعليمي المستهدف (CEFR): None

🛠️ الشكل الصحيح الذي يجب أن يرى به المدرب (SFTTrainer) هذه العينة:
[
  {
    "role": "system",
    "content": "You are an expert AI Tutor helping a student learn the word 'None' at a None level."
  }
]


In [ ]:
def peek_raw_dataset(jsonl_path):
    print("👀 جاري قراءة أول 3 أسطر من ملفك كـ (نص خام) تماماً...\n" + "="*60)
    try:
        with open(jsonl_path, 'r', encoding='utf-8') as f:
            for i in range(3):
                line = f.readline()
                if not line:
                    print(f"📝 السطر [{i+1}]: --- نهاية الملف أو سطر فارغ تماماً ---")
                else:
                    print(f"📝 السطر [{i+1}]: {repr(line)}") # باستخدام repr لإظهار الفراغات المخفية ورموز \n
    except Exception as e:
        print(f"❌ حدث خطأ أثناء قراءة الملف: {e}")

# تشغيل الفحص الخام
target_path = "/content/drive/MyDrive/Adaptive_Tutor_RL/data/processed/train_sft.jsonl"
peek_raw_dataset(target_path)

👀 جاري قراءة أول 3 أسطر من ملفك كـ (نص خام) تماماً...
📝 السطر [1]: '{"prompt": "<|im_start|>system\\nYou are an expert, patient, and adaptive English language tutor.<|im_end|>\\n<|im_start|>user\\n### WORD DATA\\n- Target Word: soil\\n- Part of Speech: noun\\n- CEFR Proficiency Level: B1\\n- Contextual Reference Examples:\\n  1. instruments for measuring soil moisture\\n  2. soil erosion\\n  3. the study of rocks and soils\\n  4. sandy/fertile soil\\n  5. rich/poor/dry/wet soil\\n  6. acid/alkaline soil\\n  7. clay soil\\n  8. moisture in the soil\\n  9. She dug the compost into the soil.\\n\\n### PEDAGOGICAL TASK\\nAct as an adaptive tutor. Initiate the structured dialogue from the training data to teach the student the word \'soil\' naturally, adhering to the provided CEFR level constraints.<|im_end|>\\n<|im_start|>assistant\\n", "completion": "Teacher: Hi, Xiao Ming. Can you tell me what the word \\"soil\\" means?\\n\\nStudent: Hi, Teacher. I\'m not sure, Teacher. I think it might b

التقييم الفني النهائي: لماذا انهار النموذج سابقاً؟
البيانات مجهزة مسبقاً بـ ChatML: ملفك لا يحتوي على مفاتيح منفصلة مثل word أو dialogue. هو مجهز بالكامل وصريح تحت مفتاحين فقط هما: "prompt" و "completion". والأخطر من ذلك، أن نصوص الـ prompt تحتوي بالفعل وبشكل يدوي صلب (Hardcoded) على رموز عائلة Qwen مثل <|im_start|>system و <|im_end|>.

سبب فشل الاختبار السابق (The Pajama Effect): النموذج أثناء التدريب الأول لم يتعلم كيف يجيب على كلمة عامة مثل hi. لقد تدرب رياضياً على نمط موحد وصارم للغاية: يرى نظام الحماية، ثم يرى كلمة ### WORD DATA متبوعة بالكلمة والمستوى والأمثلة، وعندها فقط يفتح عقله ليتصرف كـ Teacher. عندما اختبرناه بكلمة hi عارية، أصيب النموذج بذعر برمي؛ لأنه لم يجد الهيكل الذي اعتاد عليه، فبدأ يسحب عشوائياً من ذاكرته الصينية القديمة!

عملية اختبار جديدة

In [ ]:
%%writefile /content/drive/MyDrive/Adaptive_Tutor_RL/src/sft/test_qwen_sft.py
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

def launch_inference():
    base_model_id = "Qwen/Qwen1.5-1.8B-Chat"
    adapter_dir = "/content/drive/MyDrive/Adaptive_Tutor_RL/models/qwen_sft_final"

    print("📥 Loading Tokenizer and Base Model...")
    tokenizer = AutoTokenizer.from_pretrained(adapter_dir, trust_remote_code=True)

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True
    )

    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        torch_dtype=torch.bfloat16
    )

    print("🔗 Fusing LoRA Weights with Base Model...")
    model = PeftModel.from_pretrained(base_model, adapter_dir)
    model.eval()

    print("\n🎓 AI Tutor Simulating Environment Ready! Type 'exit' to stop.\n" + "="*50)

    while True:
        print("\n📥 [Input Step] Provide a new word to test the Tutor's adaptive skills:")
        word = input("   Target Word (e.g., thesis): ")
        if word.lower() == 'exit': break

        pos = input("   Part of Speech (e.g., noun): ")
        level = input("   CEFR Level (e.g., C1): ")
        example = input("   Contextual Example sentence: ")

        # -----------------------------------------------------------------
        # 🔥 حقن الهيكل المطابق بنسبة 100% لما رآه النموذج في ملف التدريب الخاص بك
        # -----------------------------------------------------------------
        structured_prompt = (
            "<|im_start|>system\nYou are an expert, patient, and adaptive English language tutor.<|im_end|>\n"
            "<|im_start|>user\n### WORD DATA\n"
            f"- Target Word: {word}\n"
            f"- Part of Speech: {pos}\n"
            f"- CEFR Proficiency Level: {level}\n"
            f"- Contextual Reference Examples:\n  1. {example}\n\n"
            f"### PEDAGOGICAL TASK\nAct as an adaptive tutor. Initiate the structured dialogue from the training data to teach the student the word '{word}' naturally, adhering to the provided CEFR level constraints.<|im_end|>\n"
            "<|im_start|>assistant\n"
        )

        inputs = tokenizer([structured_prompt], return_tensors="pt").to("cuda")

        print("\n🔮 Tutor Response (Initiating Dialogue):")
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                temperature=0.3, # درجة منخفضة جداً للالتزام بالنمط الدراسي
                top_p=0.85,
                repetition_penalty=1.2,
                do_sample=True,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id else tokenizer.eos_token_id
            )

        generated_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        print(generated_text)
        print("\n" + "="*50 + "\n")

if __name__ == "__main__":
    launch_inference()

Overwriting /content/drive/MyDrive/Adaptive_Tutor_RL/src/sft/test_qwen_sft.py


In [ ]:
!python /content/drive/MyDrive/Adaptive_Tutor_RL/src/sft/test_qwen_sft.py

📥 Loading Tokenizer and Base Model...
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights:   1% 2/291 [00:00<00:59,  4.86it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 291/291 [00:01<00:00, 190.22it/s]
🔗 Fusing LoRA Weights with Base Model...

🎓 AI Tutor Simulating Environment Ready! Type 'exit' to stop.

📥 [Input Step] Provide a new word to test the Tutor's adaptive skills:
   Target Word (e.g., thesis): synthesis
   Part of Speech (e.g., noun): noun
   CEFR Level (e.g., C1): c1
   Contextual Example sentence: what

🔮 Tutor Response (Initiating Dialogue):
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_o

In [ ]:
import torch
import json
import re
import random
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

def load_full_word_context(file_path):
    """دالة لاستخراج الكلمة مع المادة العلمية (الأمثلة والنوع) لمنع الهلوسة"""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            if not lines: return None, None

            random_line = random.choice(lines)
            data = json.loads(random_line)

            # استخراج كتلة ### WORD DATA بالكامل من ملف التدريب
            prompt_text = data.get("prompt", "")
            match = re.search(r"(### WORD DATA\n.*?)(?=\n### PEDAGOGICAL TASK)", prompt_text, re.DOTALL)

            if match:
                word_data_block = match.group(1).strip()
                # استخراج الكلمة فقط للترحيب
                word_match = re.search(r"- Target Word:\s*(.+)", word_data_block)
                target_word = word_match.group(1).strip() if word_match else "unknown"
                return target_word, word_data_block
            else:
                return "apple", "- Target Word: apple\n- Part of Speech: noun\n- CEFR Proficiency Level: A1\n- Contextual Reference Examples:\n  1. I ate a red apple."
    except Exception as e:
        print(f"Error reading file: {e}")
        return None, None

def launch_real_tutor():
    base_model_id = "Qwen/Qwen1.5-1.8B-Chat"
    adapter_dir = "/content/drive/MyDrive/Adaptive_Tutor_RL/models/qwen_sft_final"
    data_path = "/content/drive/MyDrive/Adaptive_Tutor_RL/data/processed/train_sft.jsonl"

    print("📥 Loading AI Tutor System with Knowledge Injection...")
    tokenizer = AutoTokenizer.from_pretrained(adapter_dir, trust_remote_code=True)

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )

    model = PeftModel.from_pretrained(
        AutoModelForCausalLM.from_pretrained(base_model_id, quantization_config=bnb_config, device_map="auto"),
        adapter_dir
    )
    model.eval()

    # النظام يسحب المادة العلمية الكاملة لتجنب الهلوسة
    target_word, word_data_block = load_full_word_context(data_path)

    print("\n" + "="*50)
    print(f"🎯 Target Word: '{target_word}'")
    print("="*50 + "\n")

    # توجيه صارم يحقن الأمثلة الصحيحة في دماغ النموذج
    system_prompt = (
        "You are an interactive, accurate English tutor. "
        "You must teach the student based EXACTLY on the following verified data. DO NOT invent new examples.\n\n"
        f"{word_data_block}\n\n"
        "RULES:\n"
        "1. NEVER write the student's response. You are ONLY the teacher.\n"
        "2. Keep your responses short.\n"
        "3. Evaluate the student's answer. If they don't know the word, explain it using ONLY the examples provided in the WORD DATA above."
    )

    chat_history = [{"role": "system", "content": system_prompt}]

    initial_question = f"Hi! Do you know what the word '{target_word}' means?"
    chat_history.append({"role": "assistant", "content": initial_question})
    print(f"🤖 Tutor: {initial_question}")

    while True:
        user_input = input("👤 You: ")
        if user_input.lower() == 'exit': break

        chat_history.append({"role": "user", "content": user_input})

        text = tokenizer.apply_chat_template(chat_history, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer([text], return_tensors="pt").to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=150,
                temperature=0.2, # حرارة شبه منعدمة لإجباره على النقل الحرفي من المادة العلمية
                top_p=0.85,
                repetition_penalty=1.1,
                do_sample=True,
                eos_token_id=tokenizer.eos_token_id
            )

        generated_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

        if "Student:" in generated_text:
            generated_text = generated_text.split("Student:")[0].strip()

        print(f"🤖 Tutor: {generated_text}")
        chat_history.append({"role": "assistant", "content": generated_text})

if __name__ == "__main__":
    launch_real_tutor()

📥 Loading AI Tutor System with Knowledge Injection...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]


🎯 Target Word: 'correct'

🤖 Tutor: Hi! Do you know what the word 'correct' means?
👤 You: no, i dont


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


🤖 Tutor: Great! Let's learn about "correct." It's a verb that means to fix or make right. For example, if someone says "I forgot my homework," you could say "I need to correct my homework."

Now, let's try this sentence together: "I need to correct my homework." Can you tell me what "correct" means here?

Remember, "correct" is used when you want to fix something wrong. So, if you do something wrong, you might say "I need to correct my mistake."
👤 You: i think it just make it right
🤖 Tutor: Yes, you're right. When someone says "I need to correct my homework," they mean to make everything right or fix any mistakes. So, "correct" fits perfectly in this context.

Here's how you could say it: "I need to correct my homework." Does that help clarify it for you?

Remember, "correct" is a very useful word in English. It helps us communicate clearly and accurately.
👤 You: yes
🤖 Tutor: Great job understanding "correct!" Now you've got a good grasp of this word. Keep practicing, and you'll be abl

اعادة عملية التدريب المحدثة


In [ ]:
%%writefile /content/drive/MyDrive/Adaptive_Tutor_RL/src/sft/train_qwen_v2.py
import os
import torch
import re
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    set_seed
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# ==========================================
# 1. Configuration & Paths
# ==========================================
BASE_MODEL_ID = "Qwen/Qwen1.5-1.8B-Chat"
DATA_PATH = "/content/drive/MyDrive/Adaptive_Tutor_RL/data/processed/train_sft.jsonl"
OUTPUT_DIR = "/content/drive/MyDrive/Adaptive_Tutor_RL/models/qwen_sft_v2"

set_seed(42)

def main():
    print("🚀 Initiating Adaptive Tutor SFT Training (Version 2 - Multi-turn Fixed)...\n" + "="*50)

    # ==========================================
    # 2. Load Tokenizer & Base Model (QLoRA 4-bit)
    # ==========================================
    print("📥 Loading Tokenizer and 4-bit Base Model...")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token # Qwen uses eos as pad

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, # Optimized for newer GPUs (T4/A100)
        bnb_4bit_use_double_quant=True
    )

    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )

    model = prepare_model_for_kbit_training(model)

    # ==========================================
    # 3. LoRA Configuration
    # ==========================================
    print("🧠 Configuring LoRA Adapters...")
    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"] # Full linear layer targeting for better reasoning
    )
    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()

    # ==========================================
    # 4. Data Restructuring & Loading
    # ==========================================
    print("\n📊 Loading and Reformatting Dataset...")
    dataset = load_dataset("json", data_files=DATA_PATH, split="train")

    def format_conversational_data(example):
        """
        هذه الدالة هي قلب الإصلاح:
        تأخذ النص المسطح وتفصله إلى حوار تفاعلي ليتعلم النموذج التوقف بعد كل رد
        """
        prompt_text = example['prompt']
        completion_text = example['completion']

        # 1. استخراج الـ System و User من القالب القديم
        try:
            sys_part = prompt_text.split("<|im_start|>user")[0].replace("<|im_start|>system\n", "").replace("<|im_end|>\n", "").strip()
            user_part = prompt_text.split("<|im_start|>user\n")[1].split("<|im_end|>")[0].strip()
        except:
            sys_part = "You are an expert, patient, and adaptive English language tutor."
            user_part = prompt_text

        messages = [
            {"role": "system", "content": sys_part},
            {"role": "user", "content": user_part}
        ]

        # 2. تحويل الـ Completion من "مسرحية" إلى أدوار متتابعة
        turns = completion_text.replace("\\n", "\n").split("\n\n")
        for turn in turns:
            turn = turn.strip()
            if not turn: continue
            if turn.startswith("Teacher:"):
                messages.append({"role": "assistant", "content": turn.replace("Teacher:", "").strip()})
            elif turn.startswith("Student:"):
                messages.append({"role": "user", "content": turn.replace("Student:", "").strip()})

        # 3. تطبيق قالب الدردشة الرسمي ليتم حقن رموز الإيقاف <|im_end|> بشكل صحيح
        formatted_text = tokenizer.apply_chat_template(messages, tokenize=False)
        return {"text": formatted_text}

    # تطبيق الدالة على كامل البيانات
    mapped_dataset = dataset.map(format_conversational_data, remove_columns=dataset.column_names)

    print("✅ Dataset formatted successfully. Example of first mapped sequence:")
    print(mapped_dataset[0]['text'][:500] + "...\n")

    # ==========================================
    # 5. Training Arguments (SFT)
    # ==========================================
    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        logging_steps=10,
        num_train_epochs=3, # كافية جداً ليتعلم نمط التوقف الجديد
        save_strategy="epoch",
        optim="paged_adamw_32bit",
        bf16=True, # bfloat16 لثبات التدريب
        max_grad_norm=0.3,
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
        report_to="none" # إغلاق wandb لتجنب الإزعاج
    )

    # ==========================================
    # 6. SFT Trainer Initialization
    # ==========================================
    print("🏋️‍♂️ Initializing SFT Trainer...")
    trainer = SFTTrainer(
        model=model,
        train_dataset=mapped_dataset,
        dataset_text_field="text",
        max_seq_length=1536, # مساحة كافية لاستيعاب الأمثلة والحوار الممتد
        tokenizer=tokenizer,
        args=training_args,
    )

    # ==========================================
    # 7. Start Training & Save
    # ==========================================
    print("🔥 Starting Training V2...")
    trainer.train()

    print(f"\n💾 Saving Final V2 Model to {OUTPUT_DIR}...")
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print("✅ Training Complete and Model Saved Successfully!")

if __name__ == "__main__":
    main()

In [ ]:
!python /content/drive/MyDrive/Adaptive_Tutor_RL/src/sft/train_qwen_v2.py

هذه الشيفرة يجب التحقق من انها تحتوي على كل الميزات في القائمة التي تم الاتفاق عليها

يجب مراجعتها قبل تنفيذ عملية التدريب الجديدة مع الانتباه الى الاسماء والخيارات الاخرى


In [ ]:
%%writefile /content/drive/MyDrive/Adaptive_Tutor_RL/src/sft/train_qwen_b.py
import os
import torch
import re
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    EarlyStoppingCallback,
    set_seed
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer

# تثبيت العشوائية لضمان استقرار النتائج
set_seed(42)

# =====================================================================
# 1. الإعدادات الثابتة ومعاملات التشغيل المدمجة (Hyperparameters)
# =====================================================================
BASE_MODEL_ID = "Qwen/Qwen1.5-1.8B-Chat"
DATA_PATH = "/content/drive/MyDrive/Adaptive_Tutor_RL/data/processed/train_sft.jsonl"
OUTPUT_DIR = "/content/drive/MyDrive/Adaptive_Tutor_RL/models/qwen_sft_final_b"

EPOCHS = 5
LEARNING_RATE = 2e-4
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4
MAX_LENGTH = 1280

def main():
    print(f"🚀 Initiating Ultimate Production Pipeline (Fixed for Latest TRL)...")
    print(f"📦 Targeted Architecture: {BASE_MODEL_ID} | Total Epochs: {EPOCHS}\n" + "="*60)

    # 2. تحميل الـ Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token

    # 3. تكوين تكميم النموذج لحماية الذاكرة (4-bit QLoRA)
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True
    )

    print("📥 Loading Quantized Base Model...")
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
    base_model = prepare_model_for_kbit_training(base_model)

    # 4. إعداد الـ LoRA الشامل لجميع الطبقات الخطية
    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    )
    model = get_peft_model(base_model, peft_config)
    model.print_trainable_parameters()

    # 5. دالة إعادة الهيكلة للحوار المتعدد وفرض اللغة الإنجليزية
    def format_conversational_data(example):
        prompt_text = example['prompt']
        completion_text = example['completion']

        try:
            user_part = prompt_text.split("<|im_start|>user\n")[1].split("<|im_end|>")[0].strip()
        except:
            user_part = prompt_text

        messages = [
            {"role": "system", "content": "You are an expert, patient, and adaptive English language tutor. You must conduct the entire dialogue strictly in English."},
            {"role": "user", "content": user_part}
        ]

        turns = completion_text.replace("\\n", "\n").split("\n\n")
        for turn in turns:
            turn = turn.strip()
            if not turn: continue
            if turn.startswith("Teacher:"):
                messages.append({"role": "assistant", "content": turn.replace("Teacher:", "").strip()})
            elif turn.startswith("Student:"):
                messages.append({"role": "user", "content": turn.replace("Student:", "").strip()})

        return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

    # 6. تحميل البيانات وتقسيمها
    print("\n📊 Preprocessing Dataset...")
    raw_dataset = load_dataset("json", data_files=DATA_PATH, split="train")
    split_dataset = raw_dataset.train_test_split(test_size=0.1, seed=42)

    train_dataset = split_dataset["train"].map(format_conversational_data, remove_columns=raw_dataset.column_names)
    eval_dataset = split_dataset["test"].map(format_conversational_data, remove_columns=raw_dataset.column_names)

    print(f"✅ Data Formatted. Train samples: {len(train_dataset)} | Validation samples: {len(eval_dataset)}")

    # 7. التكوين الحديث لـ SFTConfig (تنظيف التحذيرات الجانبية)
    training_args = SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        learning_rate=LEARNING_RATE,
        logging_steps=10,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        bf16=True,
        optim="paged_adamw_32bit",
        report_to="none",
        dataset_text_field="text",
        max_length=MAX_LENGTH
    )

    # 8. بناء المدرب وتمرير صمام التوقف الذكي
    # التغيير الجوهري هنا: استبدال tokenizer بـ processing_class للتوافق مع التحديث الأخير
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        args=training_args,
        processing_class=tokenizer, # <-- تم الإصلاح الجذري هنا
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    # 9. آلية الاستكمال التلقائي الذكي من نقاط الحفظ
    resume_checkpoint = None
    if os.path.exists(OUTPUT_DIR):
        checkpoints = [os.path.join(OUTPUT_DIR, d) for d in os.listdir(OUTPUT_DIR) if "checkpoint-" in d]
        if checkpoints:
            resume_checkpoint = max(checkpoints, key=os.path.getmtime)
            print(f"🔄 Checkpoint detected! Automatically resuming training from: {resume_checkpoint}")

    # 10. إطلاق محرك التدريب
    print("\n🔥 Launching Engine...")
    trainer.train(resume_from_checkpoint=resume_checkpoint)

    # 11. تأمين وحفظ الأوزان النهائية للنموذج الأفضل
    print(f"\n💾 Archiving the absolute best weights to {OUTPUT_DIR}...")
    trainer.save_model(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)
    print("🎯 Supervised Fine-Tuning Process Finished Successfully!")

if __name__ == "__main__":
    main()

Overwriting /content/drive/MyDrive/Adaptive_Tutor_RL/src/sft/train_qwen_b.py


In [ ]:
!python /content/drive/MyDrive/Adaptive_Tutor_RL/src/sft/train_qwen_b.py

🚀 Initiating Ultimate Production Pipeline (Fixed for Latest TRL)...
📦 Targeted Architecture: Qwen/Qwen1.5-1.8B-Chat | Total Epochs: 5
📥 Loading Quantized Base Model...
Loading weights:   1% 2/291 [00:00<00:53,  5.42it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 291/291 [00:01<00:00, 215.14it/s]
trainable params: 14,991,360 || all params: 1,851,820,032 || trainable%: 0.8095

📊 Preprocessing Dataset...
✅ Data Formatted. Train samples: 720 | Validation samples: 80
/content/drive/MyDrive/Adaptive_Tutor_RL/src/sft/train_qwen_b.py:105: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on u

In [4]:
!pip install trl peft accelerate transformers bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.2 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


يجب اختبار النموذج المدرب الجديد

ثم ادخال مدير الحالة الى التنفيذ وانشاء اختبار جديد

In [ ]:
%%writefile /content/drive/MyDrive/Adaptive_Tutor_RL/src/sft/test_qwen_b.py
import torch
import json
import re
import random
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

def load_full_word_context(file_path):
    """دالة لاستخراج الكلمة مع المادة العلمية (الأمثلة والنوع) لمنع الهلوسة كما هي في كودك الأصلي"""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            if not lines: return None, None

            random_line = random.choice(lines)
            data = json.loads(random_line)

            # استخراج كتلة ### WORD DATA بالكامل من ملف التدريب
            prompt_text = data.get("prompt", "")
            match = re.search(r"(### WORD DATA\n.*?)(?=\n### PEDAGOGICAL TASK)", prompt_text, re.DOTALL)

            if match:
                word_data_block = match.group(1).strip()
                # استخراج الكلمة فقط للترحيب
                word_match = re.search(r"- Target Word:\s*(.+)", word_data_block)
                target_word = word_match.group(1).strip() if word_match else "unknown"
                return target_word, word_data_block
            else:
                return "apple", "- Target Word: apple\n- Part of Speech: noun\n- CEFR Proficiency Level: A1\n- Contextual Reference Examples:\n  1. I ate a red apple."
    except Exception as e:
        print(f"Error reading file: {e}")
        return None, None

def launch_real_tutor():
    base_model_id = "Qwen/Qwen1.5-1.8B-Chat"
    # التحديث للمجلد الجديد الذي نتج عن تدريب اليوم b
    adapter_dir = "/content/drive/MyDrive/Adaptive_Tutor_RL/models/qwen_sft_final_b"
    data_path = "/content/drive/MyDrive/Adaptive_Tutor_RL/data/processed/train_sft.jsonl"
    save_log_dir = "/content/drive/MyDrive/Adaptive_Tutor_RL/outputs"

    print("📥 Loading AI Tutor System with Knowledge Injection (Model B)...")
    tokenizer = AutoTokenizer.from_pretrained(adapter_dir, trust_remote_code=True)

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )

    model = PeftModel.from_pretrained(
        AutoModelForCausalLM.from_pretrained(base_model_id, quantization_config=bnb_config, device_map="auto"),
        adapter_dir
    )
    model.eval()

    # النظام يسحب المادة العلمية الكاملة لتجنب الهلوسة
    target_word, word_data_block = load_full_word_context(data_path)

    print("\n" + "="*50)
    print(f"🎯 Target Word: '{target_word}'")
    print("="*50 + "\n")

    # توجيه صارم يحقن الأمثلة الصحيحة ويفرض الإنجليزية المطلقة ويرفض أي لغة أخرى
    system_prompt = (
        "You are an interactive, accurate English tutor. "
        "You must teach the student based EXACTLY on the following verified data. DO NOT invent new examples.\n\n"
        f"{word_data_block}\n\n"
        "RULES:\n"
        "1. NEVER write the student's response. You are ONLY the teacher.\n"
        "2. Keep your responses short.\n"
        "3. Evaluate the student's answer. If they don't know the word, explain it using ONLY the examples provided in the WORD DATA above.\n"
        "4. STRICT ENGLISH ONLY: You must conduct 100% of the conversation in English. If the student writes in any language other than English (e.g., Arabic), you must strictly reject it and reply: 'Please respond only in English.'"
    )

    chat_history = [{"role": "system", "content": system_prompt}]

    initial_question = f"Hi! Do you know what the word '{target_word}' means?"
    chat_history.append({"role": "assistant", "content": initial_question})
    print(f"🤖 Tutor: {initial_question}")

    while True:
        user_input = input("👤 You: ").strip()

        # عند كتابة exit يتم الخروج وحفظ السجل فوراً
        if user_input.lower() == 'exit':
            print("\n💾 Saving chat history logs...")
            break

        if not user_input:
            continue

        chat_history.append({"role": "user", "content": user_input})

        text = tokenizer.apply_chat_template(chat_history, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer([text], return_tensors="pt").to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=150,
                temperature=0.2, # حرارة شبه منعدمة لإجباره على النقل الحرفي من المادة العلمية
                top_p=0.85,
                repetition_penalty=1.1,
                do_sample=True,
                eos_token_id=tokenizer.eos_token_id
            )

        generated_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

        if "Student:" in generated_text:
            generated_text = generated_text.split("Student:")[0].strip()

        print(f"🤖 Tutor: {generated_text}")
        chat_history.append({"role": "assistant", "content": generated_text})

    # --- آلية حفظ السجل المستمر عند نهاية الحوار ---
    os.makedirs(save_log_dir, exist_ok=True)
    log_file_path = os.path.join(save_log_dir, "tutor_b_session_log.json")
    with open(log_file_path, "w", encoding="utf-8") as f:
        json.dump(chat_history, f, ensure_ascii=False, indent=4)

    print(f"✅ Log successfully compiled and saved to:\n📂 {log_file_path}")
    print("👋 Goodbye!")

if __name__ == "__main__":
    launch_real_tutor()

Overwriting /content/drive/MyDrive/Adaptive_Tutor_RL/src/sft/test_qwen_b.py


In [ ]:
!python /content/drive/MyDrive/Adaptive_Tutor_RL/src/sft/test_qwen_b.py

📥 Loading AI Tutor System with Knowledge Injection (Model B)...
Loading weights: 100% 291/291 [00:01<00:00, 220.53it/s]

🎯 Target Word: 'whenever'

🤖 Tutor: Hi! Do you know what the word 'whenever' means?
👤 You: i dont
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
🤖 Tutor: Great! Let's look at this together. The word "whenever" is a conjunction that we use to show time or place when something happens. For example, "You can ask for help whenever you need it."
👤 You: i dont got it 
🤖 Tutor: Okay, let me give you an example. How about "I'll be back in five minutes"? That shows how long you will stay there. Does that help clarify it?
👤 You: what do you mean , how the word whenver relate to you will ack 
🤖 Tutor: Whenver means whenever, right? So, whenever you want to say something will h

هناك شيفرة جديدة للاختبار ومن ثم نقوم باضافة مدير الحالة

In [2]:
%%writefile /content/drive/MyDrive/Adaptive_Tutor_RL/src/sft/test_qwen_b_2.py
import torch
import json
import re
import random
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

def load_full_word_context(file_path):
    """استخراج الكلمة والمادة العلمية بدقة من ملف التدريب لمنع الهلوسة"""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
            if not lines: return None, None

            random_line = random.choice(lines)
            data = json.loads(random_line)

            prompt_text = data.get("prompt", "")
            match = re.search(r"(### WORD DATA\n.*?)(?=\n### PEDAGOGICAL TASK)", prompt_text, re.DOTALL)

            if match:
                word_data_block = match.group(1).strip()
                word_match = re.search(r"- Target Word:\s*(.+)", word_data_block)
                target_word = word_match.group(1).strip() if word_match else "unknown"
                return target_word, word_data_block
            else:
                return "whenever", "- Target Word: whenever\n- Part of Speech: conjunction\n- CEFR Level: B1\n- Contextual Reference Examples:\n  1. You can ask for help whenever you need it."
    except Exception as e:
        print(f"Error reading file: {e}")
        return "whenever", "- Target Word: whenever"

def launch_real_tutor_v2():
    base_model_id = "Qwen/Qwen1.5-1.8B-Chat"
    adapter_dir = "/content/drive/MyDrive/Adaptive_Tutor_RL/models/qwen_sft_final_b"
    data_path = "/content/drive/MyDrive/Adaptive_Tutor_RL/data/processed/train_sft.jsonl"
    save_log_dir = "/content/drive/MyDrive/Adaptive_Tutor_RL/outputs"

    print("📥 Loading AI Tutor System - V2 [Strict Anchoring + Greedy Mode]...")
    tokenizer = AutoTokenizer.from_pretrained(adapter_dir, trust_remote_code=True)

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )

    model = PeftModel.from_pretrained(
        AutoModelForCausalLM.from_pretrained(base_model_id, quantization_config=bnb_config, device_map="auto"),
        adapter_dir
    )
    model.eval()

    target_word, word_data_block = load_full_word_context(data_path)

    print("\n" + "="*50)
    print(f"🎯 Target Word for Test 2: '{target_word}'")
    print("="*50 + "\n")

    # توجيهات الإصدار الثاني: التركيز المطلق على الكلمة المستهدفة والإنجليزية الصارمة
    system_prompt = (
        "You are an interactive, highly accurate English tutor.\n"
        f"Current Topic: You are teaching the specific word: '{target_word}'.\n"
        f"You must base your explanations EXACTLY on this data:\n{word_data_block}\n\n"
        "RULES:\n"
        "1. NEVER write the student's response. You are ONLY the teacher.\n"
        "2. Keep responses short and educational.\n"
        "3. NEVER lose focus on the target word. Every response must guide the student back to understanding this word.\n"
        "4. STRICT ENGLISH ONLY: Conduct 100% of the conversation in English. Reject any other language instantly.\n"
        "5. DO NOT invent outside examples. Use the ones provided above."
    )

    chat_history = [{"role": "system", "content": system_prompt}]

    initial_question = f"Hi! Do you know what the word '{target_word}' means?"
    chat_history.append({"role": "assistant", "content": initial_question})
    print(f"🤖 Tutor (V2): {initial_question}")

    while True:
        user_input = input("👤 You: ").strip()

        if user_input.lower() == 'exit':
            print("\n💾 Archiving Test 2 chat logs...")
            break

        if not user_input:
            continue

        # فحص محلي فوري لمنع أي لغة غير الإنجليزية
        if bool(re.search(r'[\u0600-\u06FF]', user_input)):
            print("🤖 Tutor (V2): Please respond only in English.")
            chat_history.append({"role": "user", "content": user_input})
            chat_history.append({"role": "assistant", "content": "Please respond only in English."})
            continue

        chat_history.append({"role": "user", "content": user_input})

        text = tokenizer.apply_chat_template(chat_history, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer([text], return_tensors="pt").to("cuda")

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=150,
                do_sample=False,          # الطور الحتمي لمنع الهلوسة والشرود الذهني للنموذج
                repetition_penalty=1.2,    # لمنع الإجابات الدائرية المكررة
                eos_token_id=tokenizer.eos_token_id
            )

        generated_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

        if "Student:" in generated_text:
            generated_text = generated_text.split("Student:")[0].strip()

        print(f"🤖 Tutor (V2): {generated_text}")
        chat_history.append({"role": "assistant", "content": generated_text})

    # حفظ السجل المخصص للإصدار الثاني
    os.makedirs(save_log_dir, exist_ok=True)
    log_file_path = os.path.join(save_log_dir, "tutor_b_2_session_log.json")
    with open(log_file_path, "w", encoding="utf-8") as f:
        json.dump(chat_history, f, ensure_ascii=False, indent=4)

    print(f"✅ Test 2 Log saved to: {log_file_path}\n👋 Goodbye!")

if __name__ == "__main__":
    launch_real_tutor_v2()

Writing /content/drive/MyDrive/Adaptive_Tutor_RL/src/sft/test_qwen_b_2.py


In [5]:
!python /content/drive/MyDrive/Adaptive_Tutor_RL/src/sft/test_qwen_b_2.py

📥 Loading AI Tutor System - V2 [Strict Anchoring + Greedy Mode]...
model.safetensors: 100% 3.67G/3.67G [00:41<00:00, 89.0MB/s]
Loading weights: 100% 291/291 [00:11<00:00, 24.73it/s]
generation_config.json: 100% 206/206 [00:00<00:00, 1.28MB/s]

🎯 Target Word for Test 2: 'adjustment'

🤖 Tutor (V2): Hi! Do you know what the word 'adjustment' means?
👤 You: no
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
🤖 Tutor (V2): Great question! An "adjustment" is something that helps make things fit better or more comfortable. For example, when we want our clothes to look good on us, we might need to adjust them a little bit before wearing them. Does that help clarify it for you?
👤 You: not at all 
🤖 Tutor (V2): No problem at all! Let me know if you have any further questions about this word.
👤 You

مرحلة التوثيق

تم الانتهاء من مرحلتي توليد وتنظيف البيانات ومرحلة بناء المولد وتدريبه SFT ضمن خيارات واعدادات مناسبة
كما تم اختباره للتأكد من مخرجاته